<center>
    <img src="https://rockborne.com/wp-content/uploads/2021/07/LandingPage-Header-RED-CENTRE.jpg" width="900" alt="logo"  />
</center>

# Correlation Analysis

*Session 4 · Notebook 04 · Lecture · Student version*

## Overview

Correlation measures how two variables move together. This notebook covers what correlation is and is not, the three main coefficients (**Pearson**, **Spearman** and **Kendall**) and when to use each, how to build and read **correlation matrices** and heatmaps (including for categorical variables), and the traps that catch people out: non-linear relationships, outliers, confounders and the cardinal rule that **correlation does not imply causation**.

The concepts are general purpose; a dedicated section shows how they map onto risk analysis, and the exercises put them to work on real data.

## Learning Objectives

By the end of this notebook you will be able to:

- Explain what a correlation coefficient measures (strength and direction, range -1 to +1).
- Compute and choose between Pearson, Spearman and Kendall correlation.
- Build, mask and interpret a correlation matrix and heatmap, including encoded categoricals.
- Recognise when correlation misleads: non-linearity, outliers, confounders, spurious links.
- State clearly why correlation does not, on its own, establish causation.

## Prerequisites

- Notebook 04_01 (distributions, skew) and 04_03 (which used correlation of missingness).
- Session 2/3 pandas (`corr`, selecting columns, `groupby`).

## Index

1. [Why this matters for risk analysis](#sec1)
2. [What correlation measures](#sec2)
3. [The three correlation coefficients](#sec3)
4. [Correlation matrices and visualisation](#sec4)
5. [When correlation misleads (and causation)](#sec5)
6. [Application: a correlation analysis workflow](#sec6)
7. [Exercises](#exercises)
8. [Additional Exercises](#additional)
9. [Challenge](#challenge)
10. [Key Takeaways](#takeaways)
11. [Further Reading](#reading)

<a id="setup"></a>
# Section 0: Setup

The teaching sections use a **synthetic Heritage Brew Collective dataset**: daily operating figures for two branches of the coffee chain (the same business we met in the data-quality notebook). We generate it with a function so we know exactly which relationships are built in, which makes it ideal for learning what each correlation coefficient does. Section 6 and the exercises use real data: the seaborn **penguins** dataset and `superstore.csv`.

**Documentation:** [scipy.stats](https://docs.scipy.org/doc/scipy/reference/stats.html) - the correlation functions used throughout (`pearsonr`, `spearmanr`, `kendalltau`).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import pearsonr, spearmanr, kendalltau
from sklearn.preprocessing import LabelEncoder
from IPython.display import display

sns.set_theme(style='whitegrid')
np.random.seed(42)

def generate_cafe_data(days_per_branch=180):
    """Synthetic daily operations for two Heritage Brew Collective branches.
    Relationships are engineered: foot traffic drives revenue (strong +); longer waits
    lower satisfaction (strong -); temperature is unrelated to revenue (near 0); and the
    price/units relationship is positive across branches but negative within each one
    (a confounder, Simpson's paradox)."""
    specs = {'Downtown': dict(traffic=320, price=5.0, units=300),
             'Suburb':   dict(traffic=180, price=3.5, units=170)}
    frames = []
    for branch, s in specs.items():
        n = days_per_branch
        foot = np.random.normal(s['traffic'], 40, n).clip(20)
        revenue = 50 + 4.5*foot + np.random.normal(0, 120, n)
        wait = 2 + 0.03*foot + np.random.normal(0, 1.0, n)
        satisfaction = (9.5 - 0.45*wait + np.random.normal(0, 0.4, n)).clip(1, 10)
        marketing = np.random.uniform(50, 500, n)
        new_cust = 120*(1 - np.exp(-marketing/120)) + np.random.normal(0, 4, n)
        price = np.random.normal(s['price'], 0.35, n)
        units = s['units'] - 25*(price - s['price']) + np.random.normal(0, 18, n)
        temperature = np.random.normal(18, 6, n)
        frames.append(pd.DataFrame({
            'branch': branch,
            'foot_traffic': foot.round(0), 'daily_revenue': revenue.round(0),
            'avg_wait_min': wait.round(1), 'satisfaction': satisfaction.round(1),
            'marketing_spend': marketing.round(0), 'new_customers': new_cust.round(0),
            'menu_price': price.round(2), 'units_sold': units.round(0),
            'temperature': temperature.round(1)}))
    return pd.concat(frames, ignore_index=True)

cafe = generate_cafe_data()
# Seaborn source (kept for reuse with other clients); to use it, swap the read_csv line for:
# peng = sns.load_dataset('penguins').dropna().reset_index(drop=True)   # for Section 6
# Or read directly from the public S3 bucket (no local file needed):
# peng = pd.read_csv('https://rockborne-bucket-01-cbs.s3.eu-west-2.amazonaws.com/Data_Sources_CBS_Risk/Session_4/penguins.csv').dropna().reset_index(drop=True)   # for Section 6
# ...or read the paths from a config file (the local read below stays the default):
# from config import session_datasets_http
# peng = pd.read_csv(session_datasets_http["penguins"]).dropna().reset_index(drop=True)   # for Section 6
# Or from S3 with Spark, then to pandas (needs a SparkSession, e.g. on Databricks):
# peng = spark.read.csv("s3://rockborne-bucket-01-cbs/Data_Sources_CBS_Risk/Session_4/penguins.csv", header=True, inferSchema=True).toPandas().dropna().reset_index(drop=True)   # for Section 6
peng = pd.read_csv('../datasets/Session_4/penguins.csv').dropna().reset_index(drop=True)   # for Section 6
# Or read directly from the public S3 bucket (no local file needed):
# store = pd.read_csv('https://rockborne-bucket-01-cbs.s3.eu-west-2.amazonaws.com/Data_Sources_CBS_Risk/Session_4/superstore.csv', encoding='latin1')  # for exercises
# Or from S3 with Spark, then to pandas (needs a SparkSession, e.g. on Databricks):
# store = spark.read.csv("s3://rockborne-bucket-01-cbs/Data_Sources_CBS_Risk/Session_4/superstore.csv", header=True, inferSchema=True).toPandas()  # for exercises
store = pd.read_csv('../datasets/Session_4/superstore.csv', encoding='latin1')  # for exercises
print('cafe:', cafe.shape, '| penguins:', peng.shape, '| superstore:', store.shape)
cafe.head()

### The Heritage Brew Collective data dictionary

Each row is **one day at one branch** (180 days for each of the two branches, so 360 rows in total). The columns:

| Column | What it represents |
|---|---|
| `branch` | which branch the day belongs to: Downtown or Suburb |
| `foot_traffic` | number of customers who came in that day |
| `daily_revenue` | total takings that day |
| `avg_wait_min` | average time a customer waited to be served (minutes) |
| `satisfaction` | average customer satisfaction score that day (1 to 10) |
| `marketing_spend` | amount spent on marketing that day |
| `new_customers` | number of first-time customers that day |
| `menu_price` | average menu price charged that day |
| `units_sold` | number of items sold that day |
| `temperature` | outside temperature that day (degrees C) |

<a id="sec1"></a>
# Section 1: Why this matters for risk analysis

Correlation is one of the most used (and misused) tools in a risk analyst's kit.

| Use of correlation | Risk example |
|---|---|
| **Find drivers** | Which variables move with default or loss, to shortlist features for a model? |
| **Detect redundancy** | Two highly correlated inputs cause multicollinearity and unstable model coefficients; we drop one. |
| **Diversification** | Portfolio risk depends on how asset returns correlate; low correlation reduces total risk. |
| **Stress and monitoring** | Correlations that break down in a crisis (or drift over time) are themselves a risk signal. |
| **Avoiding false stories** | A spurious or confounded correlation can send a model, or a decision, badly wrong. |

The recurring danger is reading too much into a number: a correlation is a starting point for investigation, not a conclusion.

<a id="sec2"></a>
# Section 2: What correlation measures

**Definition:** correlation quantifies the **strength** and **direction** of the relationship between two variables, as a single number between -1 and +1.

**Example:** at Heritage Brew Collective, busier days (more foot traffic) bring in more revenue; the two are positively correlated.

**Analogy:** two dancers. A correlation near +1 means they move in perfect step together; near -1 means whenever one steps forward the other steps back; near 0 means they move independently.

**Explanation:** the sign is the direction and the magnitude is the strength:

- **+1:** perfect positive (as one rises, the other always rises).
- **0:** no (linear) relationship; the variables move independently.
- **-1:** perfect negative (as one rises, the other always falls).

A common rule of thumb for the absolute value:

| Absolute correlation | Strength |
|---|---|
| above 0.7 | strong |
| 0.3 to 0.7 | moderate |
| below 0.3 | weak |


### A first look at the dataset

Before measuring any correlation, we get to know the data: its size, columns, summary statistics and the shape of each variable. This is always the first step of an analysis, and it tells us which variables are continuous (suitable for Pearson) and which are skewed.

In [ ]:
# The distribution of each numeric column

In [ ]:
# A pairplot shows every pairwise relationship at once, coloured by branch

<a id="sec3"></a>
# Section 3: The three correlation coefficients

There are three main correlation coefficients, suited to different data and relationships. We introduce each with a concrete business question about the cafe, work the example, then explore further. All are available through `df.corr(method=...)` or the `scipy.stats` functions, which also return a p-value for testing whether the correlation differs from zero.

**Two families.** These coefficients fall into two families. **Pearson** is the *parametric* one: it works on the raw values and measures a linear relationship. **Spearman** and **Kendall** are the *rank-based* (non-parametric) family: they work on the ranks of the data, so they capture any monotonic relationship and are robust to outliers and skew. Because Spearman and Kendall are in the same family they usually tell the same story, whereas Pearson can differ from both when a relationship is non-linear or outlier-driven.

**Parametric vs non-parametric.** A *parametric* method assumes the data follows a particular distribution described by parameters (Pearson effectively assumes roughly normal, linear data) and works on the raw values, so it is powerful when those assumptions hold but distorted when they do not. A *non-parametric* method makes far fewer assumptions about the distribution; the rank-based Spearman and Kendall throw away the exact values and keep only the order, which is why they tolerate skew, outliers and curves.

### How the three coefficients respond to different shapes

Before the real examples, a quick illustration on synthetic data. We build four relationships and print all three coefficients for each, so you can see *why* they disagree.

**Reading it.** The linear case reads the same on all three. The curved-but-rising case is where Spearman and Kendall exceed Pearson, because they need only the order to be consistent, not the shape to be a straight line. The U-shape returns near zero from all three: a correlation coefficient detects only a *monotonic* trend, so a real but non-monotonic relationship is invisible to it (which is why we always plot). The random case is near zero, as expected.

## 3.1 Pearson correlation (r)

**Definition:** Pearson's r measures the strength of a **linear** relationship between two continuous variables (how well the points fit a straight line).

**Example:** foot traffic vs daily revenue at the cafe, which line up almost on a straight line.

**Analogy:** drawing the best straight line through a scatter and asking how tightly the points hug it.

**Explanation - when to use:** both variables continuous, the relationship roughly linear, the data roughly normal, and no severe outliers (because r is sensitive to them). **Syntax:** `df[['a','b']].corr(method='pearson')` or `pearsonr(a, b)`.

**Critical concept: correlation does NOT imply causation** (we return to this in Section 5).

**Business case:** management wants to know whether busier days really do bring in more money. We analyse the correlation between `foot_traffic` and `daily_revenue`.

**What are `r` and `p`?** `pearsonr` returns two numbers. `r` is the **correlation coefficient** itself (-1 to +1): the strength and direction of the linear relationship. `p` is the **p-value** from a hypothesis test (see 04_02) whose null hypothesis is "the true correlation is zero"; it is the probability of seeing a correlation at least this strong if the two variables were actually unrelated. A small p (say < 0.05) means the correlation is statistically significant (unlikely to be a fluke of this sample), while a large p means we cannot rule out zero. Note that with many rows even a tiny, unimportant `r` can have a small `p`, so read them together: `r` for size, `p` for significance.

**Exploring further:** foot traffic is not the only thing that moves with revenue. We can rank how strongly every other numeric feature correlates with `daily_revenue` to see what else is worth a closer look.

In [ ]:
# The next strongest is avg_wait_min: on busy (high-revenue) days, queues are longer.

## 3.2 Spearman correlation (rho)

**Definition:** Spearman's rho is Pearson's r computed on the **ranks** of the data. It measures any **monotonic** relationship (consistently increasing or decreasing), not just a straight line.

**Example:** the cafe's marketing spend vs new customers: spending more always brings more new customers, but with diminishing returns, so the relationship curves upward while never turning back down.

**Analogy:** a dimmer switch. Turning it up always makes the room brighter, though not by equal amounts at each turn. Spearman captures that steady 'always brighter' trend even when the increase is not a straight line.

**Explanation - when to use:** ordinal data, non-linear-but-monotonic relationships, skewed data, or when outliers would distort Pearson (ranks are robust to extreme values). **Syntax:** `df.corr(method='spearman')` or `spearmanr(a, b)`.

**Business case:** the marketing team knows that spending more attracts more new customers, but suspects each extra pound works less hard than the last. This is **diminishing returns**: every additional pound of marketing brings a few fewer new customers than the one before, so the curve rises but flattens off. We analyse `marketing_spend` vs `new_customers`, both already in the cafe data, and plot it first because the relationship is likely to curve.

In [ ]:
# Plot the relationship two ways: raw values (curved) and ranks (what Spearman sees)

**Reading the two graphs.** The left panel curves (diminishing returns), which is why Pearson is only about 0.88. The right panel replaces every value with its **rank** (its position in sorted order); because the relationship always rises, the ranks line up on a near-straight diagonal, and that straight rank-line is exactly what Spearman measures, hence its higher value (about 0.92). Points hugging the diagonal mean the order is well preserved; scatter away from it would mean the two variables disagree on the ordering.

## 3.3 Kendall correlation (tau)

**Definition:** Kendall's tau is a rank-based measure built on **concordant vs discordant pairs**: for every pair of observations it asks whether they are ordered the same way on both variables. Tau is (concordant minus discordant) divided by the number of pairs, so it reads directly as a net rate of agreement.

**Example:** two inspectors each rate the cafe's drinks on a 1 to 5 scale; tau measures how often they agree on which of any two drinks is better.

**Analogy:** for every pair of items, a vote of 'agree' (both rank it the same way) or 'disagree'; tau is the net score.

**Explanation - when to use:** like Spearman it is rank-based and robust, but it is specifically preferred for **small samples** and **ordinal data with many tied ranks**, where it behaves better than Spearman. **Syntax:** `df.corr(method='kendall')` or `kendalltau(a, b)`.

**Business case (a dedicated small dataset):** two quality inspectors independently score 12 of the cafe's drinks on a 1 to 5 scale. The data is small, ordinal and full of ties, which is exactly where Kendall's tau is the right tool. We want to know how strongly the inspectors agree.

We also compute Spearman here, not because Kendall needs it, but to confirm that the two rank-based measures agree, and to see that tau is the smaller of the two.

In [ ]:
# A quick visual: where the inspectors agree, points sit on the diagonal

### Choosing a coefficient

| Coefficient | Measures | Best when | Robust to outliers? |
|---|---|---|---|
| **Pearson (r)** | linear relationship | continuous, linear, roughly normal | no |
| **Spearman (rho)** | monotonic relationship | ordinal, non-linear-monotonic, skewed | yes |
| **Kendall (tau)** | concordance of pairs | small samples, many ties | yes |

A practical habit: if Pearson and Spearman disagree a lot, the relationship is probably non-linear or outlier-driven, so plot it before trusting either number.

**Best practices**
- **Always plot first.** A coefficient is a summary; the scatter shows the shape (see Anscombe in Section 5).
- **Default to Spearman when unsure** on skewed, ordinal or outlier-prone data; it degrades gracefully.
- **Compare Pearson and Spearman.** A big gap flags non-linearity or outliers; investigate rather than pick the bigger number.
- **Report the p-value and the sample size** alongside `r`; significance is not the same as importance.
- **Never read causation into a correlation** (Section 5).

### Try it yourself

For the cafe's `avg_wait_min` and `satisfaction`, print the Pearson, Spearman and Kendall correlations. They should all be strongly negative; is Kendall the smallest in magnitude?

In [ ]:
# Your turn. Write your solution here:

<a id="sec4"></a>
# Section 4: Correlation matrices and visualisation

**Definition:** a **correlation matrix** is a table of the correlation between every pair of numeric columns. A **heatmap** colours that table so the strong relationships jump out.

**Example:** the matrix of the cafe's daily operating metrics.

**Analogy:** a mileage chart between cities, but the 'distance' is how strongly two variables move together.

**Explanation:** `df.corr(numeric_only=True)` builds the matrix (default Pearson; pass `method='spearman'` or `'kendall'`), and `sns.heatmap(..., annot=True)` draws it.

### A cleaner heatmap: mask the upper triangle

A correlation matrix is symmetric and its diagonal is always 1, so half of it is redundant. Masking the upper triangle removes the clutter and makes the result easier to read.

In [ ]:
# True for the upper triangle

### Including categorical variables

Correlation needs numbers, so categorical columns (like `branch`) must be encoded first. Two common approaches:

- **Label encoding:** map each category to an integer. Quick, but it invents an order that may not exist, so read those correlations with care.
- **One-hot encoding:** create a 0/1 column per category. No false ordering, and each category's relationship is visible separately.

In [ ]:
# Label encoding: one integer column per category
# One-hot encoding: a 0/1 column per category (no invented ordering)

### A proper measure for two categorical variables: Cramer's V

Encoding then correlating (above) is a workaround. When **both** variables are categorical, the principled measure of association is **Cramer's V**. It is built on the chi-square test of independence and is scaled from **0** (no association) to **1** (perfect association), so it reads like a correlation for categorical data. Use it instead of forcing categories into a Pearson or Spearman coefficient.

The cafe has just one categorical column (`branch`), so to demonstrate we derive a few more by banding continuous metrics: `traffic_band` (Low/Medium/High foot traffic), `wait_band` (Short/Medium/Long waits) and a random `promo` flag (a control that should be unrelated to everything). We then compute Cramer's V between every pair and show it as a heatmap.

In [ ]:
# The cafe has one categorical (branch); derive a few more to demonstrate

**How to read this heatmap.** Each cell is a single Cramer's V for a *pair of variables*, not per category value: you see `branch` vs `wait_band` as one number (about 0.80), not `Downtown` vs `Long`. It answers "how strongly are these two variables associated overall?" on a 0 (unrelated) to 1 (one perfectly predicts the other) scale. The diagonal is 1 (each variable with itself), and the random `promo` flag sits near 0 against everything, as designed. To see *which* category combinations drive a strong association, look at the underlying contingency table:

In [ ]:
# Which wait bands does each branch fall into? (the detail behind branch x wait_band = 0.80)

### Try it yourself

Build a **Spearman** correlation matrix of the cafe's numeric columns and draw it as a heatmap (`cafe.corr(method='spearman', numeric_only=True)`).

In [ ]:
# Your turn. Write your solution here:

<a id="sec5"></a>
# Section 5: When correlation misleads (and causation)

A single correlation number hides a lot. Three traps matter most, and all share one cure: **plot the data and think about the context**.

## 5.1 Correlation does not imply causation

Two variables can correlate because one causes the other, because a third variable drives both, or by pure coincidence. Ice-cream sales correlate with drownings, but neither causes the other; hot weather drives both. A correlation is evidence to investigate, never proof of cause.

## 5.2 Always plot: Anscombe's quartet

Anscombe's quartet is four small datasets with **almost identical** means, variances and Pearson correlation (about 0.82), yet wildly different shapes: one is linear, one is curved, one is a straight line dragged by a single outlier, and one is a vertical stack. The summary number alone would call them the same; the plots show they are nothing alike.

In [ ]:
# Seaborn source (kept for reuse with other clients); to use it, swap the read_csv line for:
# ans = sns.load_dataset('anscombe')
# Or read directly from the public S3 bucket (no local file needed):
# ans = pd.read_csv('https://rockborne-bucket-01-cbs.s3.eu-west-2.amazonaws.com/Data_Sources_CBS_Risk/Session_4/anscombe.csv')
# ...or read the paths from a config file (the local read below stays the default):
# from config import session_datasets_http
# ans = pd.read_csv(session_datasets_http["anscombe"])
# Or from S3 with Spark, then to pandas (needs a SparkSession, e.g. on Databricks):
# ans = spark.read.csv("s3://rockborne-bucket-01-cbs/Data_Sources_CBS_Risk/Session_4/anscombe.csv", header=True, inferSchema=True).toPandas()
ans = pd.read_csv('../datasets/Session_4/anscombe.csv')
print('Pearson r within each dataset:')
print(ans.groupby('dataset').apply(lambda g: pearsonr(g['x'], g['y'])[0], include_groups=False).round(3))

sns.lmplot(data=ans, x='x', y='y', col='dataset', col_wrap=2, height=3, ci=None,
           line_kws={'color': 'red'})
plt.show()

## 5.3 Confounders and Simpson's paradox

A **confounder** is a hidden variable that drives the relationship you see. Sometimes controlling for it even **reverses** the correlation, which is Simpson's paradox. At the cafe, `menu_price` and `units_sold` look **positively** correlated overall, which seems to say 'charge more, sell more'. Let us look at that overall relationship first.

But that overall trend is misleading; the **branch** is confounding it. The Downtown branch charges more *and* sells more (a busier location), which creates a positive overall pattern. When we split by branch and look **within** each one, the real demand relationship appears: higher price, fewer units.

<a id="sec6"></a>
# Section 6: Application - a correlation analysis workflow

A reusable routine for exploring relationships in a **real** dataset (the penguins): (1) get a visual overview with a pairplot, (2) build the correlation matrix, (3) compare Pearson with Spearman to spot non-linearity or outlier effects, (4) rank the strongest pairs, (5) visualise the top pair, and (6) sanity-check it against a possible confounder. This is the template to reach for whenever you meet a new dataset.

In [ ]:
# 1. Visual overview: every numeric pair at once, coloured by species

In [ ]:
# 2 and 3. Pearson and Spearman matrices side by side

In [ ]:
# 4. Rank every pair by absolute Pearson correlation

In [ ]:
# 5. Visualise the strongest pair, and 6. check it within each species (confounder)

<a id="exercises"></a>
# Section 7: Exercises

These exercises use the real `penguins` data from Section 0.

### Exercise 1: Pick the right coefficient

For penguins `bill_length_mm` vs `body_mass_g`, compute the Pearson, Spearman and Kendall correlations. Do they broadly agree, and what does that tell you about the shape of the relationship?

In [ ]:
# Your turn. Write your solution here:

### Exercise 2: Correlation matrix and masked heatmap

Build a Pearson correlation matrix of the penguins numeric columns and draw a masked (lower-triangle) heatmap.

In [ ]:
# Your turn. Write your solution here:

<a id="additional"></a>
## Additional Exercises

These use the real `superstore` data.

### Exercise A1: Pearson vs Spearman on skewed data

The superstore `Sales` column is highly right-skewed. For `Sales` vs `Profit`, compute both Pearson and Spearman. Which is more trustworthy here, and why?

In [ ]:
# Your turn. Write your solution here:

### Exercise A2: Superstore correlation matrix

Build a Pearson correlation matrix of the superstore numeric columns (`['Sales', 'Quantity', 'Discount', 'Profit']`) and draw a masked heatmap.

In [ ]:
# Your turn. Write your solution here:

### Exercise A3: Cramer's V on two categoricals

Using the `cramers_v` function, measure the association between penguin `species` and `island`, and between `species` and `sex`. Which pair is strongly associated, and which is essentially unrelated?

<a id="challenge"></a>
## Challenge (optional): a full correlation study

Investigate the drivers of penguin body mass. Work through the four parts; the coach answer shows one complete solution.

**Part 1:** Build a Pearson correlation matrix of the numeric columns and identify the two variables most correlated with `body_mass_g`.

**Part 2:** For the strongest driver, compare Pearson, Spearman and Kendall, and say whether the relationship looks linear.

**Part 3:** Plot that driver against `body_mass_g`, coloured by `species`, and check whether the relationship holds within each species (a confounder check).

**Part 4:** Write a two-sentence conclusion, being careful about causation.

In [ ]:
# Your turn. Write your solution here:

<a id="takeaways"></a>
## Key Takeaways

| Concept / command | What it does |
|---|---|
| Correlation range -1 to +1 | Sign is direction, magnitude is strength |
| `corr(method='pearson')` / `pearsonr` | Linear relationship; sensitive to outliers |
| `corr(method='spearman')` / `spearmanr` | Monotonic relationship on ranks; robust |
| `corr(method='kendall')` / `kendalltau` | Concordance of pairs; good for small n and ties |
| `sns.heatmap(corr, annot=True)` | Visualise a correlation matrix |
| `np.triu` mask | Hide the redundant upper triangle |
| LabelEncoder / `get_dummies` | Encode categoricals before correlating |
| Anscombe's quartet | Same r, different shapes: always plot the data |
| Simpson's paradox | A confounder can reverse a correlation; check within groups |
| Correlation != causation | A correlation is a lead to investigate, not proof of cause |


## Conclusion

You can now measure relationships with the right coefficient (Pearson, Spearman or Kendall), build and read correlation matrices including encoded categoricals, and avoid the classic traps of non-linearity, outliers and confounders. Above all, you can state plainly why correlation is not causation. The next notebook is the capstone practical, where these tools come together in a full exploratory analysis.

<a id="reading"></a>
## Further Reading & Resources

- [pandas DataFrame.corr](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.corr.html) Pearson, Spearman and Kendall in one call.
- [SciPy correlation functions](https://docs.scipy.org/doc/scipy/reference/stats.html#correlation-functions) `pearsonr`, `spearmanr`, `kendalltau` with p-values.
- [Anscombe's quartet](https://en.wikipedia.org/wiki/Anscombe%27s_quartet) and [Simpson's paradox](https://en.wikipedia.org/wiki/Simpson%27s_paradox) background on why plotting matters.